# 📊 07｜健身房會員：分清楚「入場人次」與「會員人數」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/07_gym_groupby/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：同一位會員會出現很多次：**數列數（count）得到的是「人次」，數不重複的會員編號（nunique）才是「人數」**。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：分清楚入場人次與會員人數，最後用**一張垂直長條圖**回答「哪家分店入場人次最多？」並附上會員人數。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出分店摘要：入場人次、會員人數、平均每人入場次數。
3. **選做挑戰（單元五～六）**：觀察消費金額為什麼平均數和中位數差很多，或比較各年齡層的運動時間。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/gym_visits.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一位會員的一次入場紀錄**，期間 2025-01-01～06-30，共 3,000 列。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`紀錄編號`（一次入場）、`會員編號`（一位會員，會重複出現）➔ 只能計數；數紀錄是「人次」，數不重複會員才是「人數」。
- 🔵 **條件 (Dimension)**：`分店`、`方案`、`年齡層`（類別）；`日期`（時間）。
- 🟢 **指標 (Metric)**：`運動分鐘`（分鐘）、`消費金額`（元，多數為 0）。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：哪一家分店的入場人次最多？會員人數的排名一樣嗎？
- 📊 **分布（選做）**：消費金額大多是多少？為什麼平均數和中位數差這麼多？
- 👥 **比較＋分布（選做）**：不同年齡層，每次運動的時間分布有什麼不同？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **人次 ≠ 人數**：3,000 列是 3,000 次入場，不是 3,000 位會員；「會員人數」要用 `nunique`。
- ⚠️ **沒出現的會員數不到**：會員編號到 G200，但只有 195 位半年內有入場；這份資料無法得知那 5 位的分店，不能把「有入場的會員數」說成「全部會員數」。
- ⚠️ **方案不能直接比消費**：單次方案的消費金額含 250 元入場費，和其他方案的「加購」意義不同。
- ⚠️ **年齡層差異不推因果**：年長者平均運動時間較短，不能說「年齡造成運動時間變短」；常來的人在資料中出現較多次，也會影響平均。

### 4. 決定最適圖表
- **4 家分店比入場人次** ➔ **垂直長條圖**（Y 軸從 0 開始；柱子上同時標出會員人數）。
- **消費金額集中在哪裡（選做）** ➔ **直方圖**，加上平均線與中位數線。
- **各年齡層運動分鐘（選做）** ➔ **箱形圖**（多組比較分布，X 軸依年齡順序）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'gym_visits.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/07_gym_groupby/data/gym_visits.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料，確認一列代表什麼（貼標籤）

> 🎯 **思維重點**：一列是一次入場，不是一位會員。先數「列數」和「不重複會員數」，看兩者差多少。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/gym_visits.csv：
1. 顯示前 5 筆資料、所有欄位名稱、總列數與各欄缺值數。
2. 分別計算「紀錄編號」與「會員編號」各有幾個不同的值。
3. 找出入場次數最多的前 3 位會員與他們的次數。
4. 用文字說明：一列代表什麼？哪些欄位是標識、條件、指標？
先不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/gym_visits.csv')
print("--- 前 5 筆資料 ---")
display(df.head())
print(f"\n總列數：{len(df)}；缺值：{df.isna().sum().sum()}")
print(f"不同紀錄編號：{df['紀錄編號'].nunique()}；不同會員編號：{df['會員編號'].nunique()}")
print("\n入場次數最多的 3 位會員：")
print(df['會員編號'].value_counts().head(3))
print("\n➡️ 一列代表：一位會員的一次入場；同一位會員會出現很多次。")

### 📝 單元二【必做】：邊界檢核：count 數人次、nunique 數人數

> 🎯 **思維重點**：同一個分組，用兩種方法數：數紀錄得到「入場人次」，數不重複會員得到「會員人數」；兩者相除是「平均每人入場次數」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依「分店」分組，計算以下統計並整理成一張表：
1. 入場人次：紀錄筆數（count）。
2. 會員人數：不重複的會員編號數（nunique）。
3. 平均每人入場次數：入場人次 ÷ 會員人數，保留一位小數。
4. 另外示範一個常見錯誤：用 count 數「會員編號」當成會員人數，會得到什麼？
5. 核對：各分店會員人數加總是否等於全部不重複會員數？會員編號到 G200，實際出現幾位？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
branch_df = df.groupby('分店').agg(入場人次=('紀錄編號', 'count'), 會員人數=('會員編號', 'nunique'))
branch_df['平均每人入場次數'] = (branch_df['入場人次'] / branch_df['會員人數']).round(1)
branch_df = branch_df.sort_values('入場人次', ascending=False)
display(branch_df)

wrong = df.groupby('分店')['會員編號'].count()
print("❌ 用 count 數會員編號（其實是人次）：", wrong.to_dict())
print(f"\n各分店會員人數加總：{branch_df['會員人數'].sum()}；全部不重複會員：{df['會員編號'].nunique()}")
print("⚠️ 會員編號到 G200，但只有上表這些會員在半年內有入場；沒入場的會員不會出現在資料中。")

### 📝 單元三【必做】：提問積木【比較】— 哪家分店入場人次最多？（垂直長條圖）

> 🎯 **思維重點**：4 家分店比高低，用【垂直長條圖】；柱子高度是「人次」，在柱子上同時標註「會員人數」，避免讀者把人次當成人數。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 matplotlib 畫出各分店「入場人次」的垂直長條圖：
1. X 軸為分店，Y 軸為入場人次，依人次由高到低排列，Y 軸從 0 開始。
2. 每根柱子上方標註「XXX 人次」，第二行標註「會員 XX 人」。
3. 入場人次最多的分店用醒目顏色標示，並寫在標題中。
4. 圖表標題、座標軸全部使用繁體中文；Y 軸名稱寫「入場人次（次）」而不是「人數」。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

top_branch = branch_df['入場人次'].idxmax()
colors = ['#e74c3c' if b == top_branch else '#3498db' for b in branch_df.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(branch_df.index, branch_df['入場人次'], color=colors, width=0.55)
for bar, members in zip(bars, branch_df['會員人數']):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 10, f'{h:.0f} 人次\n會員 {members} 人', ha='center', va='bottom', fontsize=10)
plt.ylim(0, branch_df['入場人次'].max() * 1.2)
plt.title(f'上半年各分店入場人次（最多：{top_branch}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('分店')
plt.ylabel('入場人次（次）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來
- 3,000 列入場紀錄，只有 **195 位**不同會員；用 count 數會員編號會錯得到 3,000。
- 長條圖與表格都顯示**松江店**入場人次最多（913 人次、57 位會員）；新莊店最少（557 人次、37 位會員）。
- 用自己的話回答：板橋店入場 883 人次、會員 53 人，平均每人來幾次？這和「會員多」是同一件事嗎？

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出分店摘要 Excel

> 🎯 **思維重點**：報表欄名要讓人一看就知道是「次」還是「人」；附上資料期間與說明，避免誤讀。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將分店摘要匯出成 Excel：
1. 欄位：分店、入場人次、會員人數、平均每人入場次數；最後加一列「全部」（全部分店的人次、不重複會員數與平均）。
2. 存為 output/分店摘要.xlsx（若 output 目錄不存在請自動建立）。
3. 匯出後重新讀取，核對入場人次合計為 3,000、全部會員數為不重複會員數。
4. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
total = pd.DataFrame({'入場人次': [len(df)], '會員人數': [df['會員編號'].nunique()]}, index=['全部'])
total['平均每人入場次數'] = (total['入場人次'] / total['會員人數']).round(1)
summary = pd.concat([branch_df, total]).rename_axis('分店').reset_index()
display(summary)

os.makedirs('output', exist_ok=True)
output_path = 'output/分店摘要.xlsx'
summary.to_excel(output_path, index=False)
check = pd.read_excel(output_path)
print('重新讀取：分店入場人次合計', check.loc[check['分店'] != '全部', '入場人次'].sum(),
      '；全部會員數', check.loc[check['分店'] == '全部', '會員人數'].iloc[0])

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【分布】— 消費金額的平均數為什麼騙人？（直方圖）

> 🎯 **思維重點**：多數入場消費是 0，少數人花很多，平均數會被拉高；這時要同時看**中位數**與「零消費比例」。單次方案含入場費，先排除。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析「消費金額」的分布：
1. 依方案計算：入場人次、平均消費、中位數消費、零消費比例（%），並說明單次方案為什麼不能直接比較。
2. 排除單次方案後，畫出消費金額的直方圖，並畫出平均數線與中位數線（不同顏色、加圖例）。
3. 用文字說明：平均數與中位數差多少？若要向老闆報告「一般會員每次加購多少」，該用哪一個數字？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
spend = df.groupby('方案')['消費金額'].agg(入場人次='count', 平均消費='mean', 中位數消費='median',
                                          零消費比例=lambda s: (s == 0).mean() * 100).round(1)
display(spend)
print("⚠️ 單次方案的消費含 250 元入場費，和其他方案的「加購」意義不同，先排除。")

member_spend = df.loc[df['方案'] != '單次', '消費金額']
plt.figure(figsize=(8, 4.5))
plt.hist(member_spend, bins=20, color='#3498db', edgecolor='white')
plt.axvline(member_spend.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'平均數 {member_spend.mean():.1f} 元')
plt.axvline(member_spend.median(), color='#2c3e50', linestyle=':', linewidth=2, label=f'中位數 {member_spend.median():.0f} 元')
plt.title('月繳／季繳／年繳會員每次入場的加購金額分布', fontsize=13, fontweight='bold')
plt.xlabel('消費金額（元）')
plt.ylabel('入場人次')
plt.legend()
plt.show()
print(f"💡 {(member_spend == 0).mean():.0%} 的入場沒有加購；中位數是 0 元，平均數被少數高額消費拉高。")

### 📝 單元六【選做挑戰】：提問積木【比較＋分布】— 各年齡層的運動時間（箱形圖）

> 🎯 **思維重點**：多個群組比較分布，用【箱形圖】一次看中位數、範圍與離群值；年齡層有順序，X 軸要依年齡排列。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較各年齡層每次入場的「運動分鐘」：
1. 依年齡層計算入場人次、平均與中位數運動分鐘。
2. 畫出箱形圖：X 軸為年齡層（依 18-24、25-34、35-44、45-54、55+ 排序），Y 軸為運動分鐘。
3. 用文字描述各年齡層中位數的變化，並附註：這只是相關，不能說年齡「造成」運動時間變短；常來的會員在資料中出現較多次，也會影響結果。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
age_stats = df.groupby('年齡層')['運動分鐘'].agg(入場人次='count', 平均='mean', 中位數='median').reindex(age_order).round(1)
display(age_stats)

plt.figure(figsize=(9, 5))
plt.boxplot([df.loc[df['年齡層'] == a, '運動分鐘'] for a in age_order], patch_artist=True,
            boxprops=dict(facecolor='#a8dadc', color='#1d3557'), medianprops=dict(color='#e74c3c', linewidth=2))
plt.xticks(range(1, len(age_order) + 1), age_order)
plt.title('各年齡層每次入場的運動分鐘', fontsize=13, fontweight='bold')
plt.xlabel('年齡層')
plt.ylabel('運動分鐘（分鐘）')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()
print("⚠️ 邊界提醒：年齡層與運動時間只有相關；不能推論年齡造成運動時間變短。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：3,000 人次、195 位會員；松江店入場人次最多，圖上同時標出會員人數。
- [ ] **報表應用**：分店摘要 Excel 區分人次與人數，「全部」列使用不重複會員數。
- [ ] **選做挑戰**：說明平均數與中位數的差距；年齡層比較只描述相關。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。